<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session04_prompt_engineering_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۴: پرامپت‌نویسی و خروجی ساختاریافته

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session04_prompt_engineering.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import json

import pandas as pd
from llm_utils import download_data
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, Optional

reviews = pd.read_csv(download_data("reviews.csv"))
LABELS = ["positive", "negative", "neutral"]


def parse_label(text):
    text = text.strip().lower()
    return next((label for label in LABELS if label in text), "unknown")


def accuracy(classify, data=reviews):
    predictions = pd.Series([classify(t) for t in data.text], index=data.index)
    return (predictions == data.label).mean(), predictions


SYSTEM_EN = """You classify Persian customer reviews of an online electronics store.

Labels:
- positive: the customer is satisfied with the product, delivery or service.
- negative: the customer reports a problem or is dissatisfied.
- neutral: a factual statement or an opinion that is neither clearly positive nor negative.

Answer with exactly one word: positive, negative or neutral."""

EXAMPLES = [
    ("مانیتور بدون حتی یک پیکسل سوخته رسید، رنگ‌هاش هم عالیه.", "positive"),
    ("دو هفته از تاریخ تحویل گذشته و هنوز خبری از سفارشم نیست.", "negative"),
    ("این ماوس با باتری قلمی کار می‌کنه و باتریش داخل جعبه نیست.", "neutral"),
    ("کیبورد خوبیه ولی نور پس‌زمینه‌اش ضعیفه.", "neutral"),
]


def make_classifier(system, examples=()):
    def classify(text):
        messages = [{"role": "system", "content": system}]
        for example, label in examples:
            messages += [{"role": "user", "content": f"Review: {example}"},
                         {"role": "assistant", "content": label}]
        messages.append({"role": "user", "content": f"Review: {text}"})
        return parse_label(chat(client, MODEL, messages=messages, temperature=0))
    return classify


def extract(schema, system, text):
    response = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": text}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema()}})
    return schema.model_validate_json(response.choices[0].message.content)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

`SYSTEM_CLASSIFIER` را به فارسی ترجمه کنید و دقت Zero-shot و Few-shot را با نسخه انگلیسی مقایسه کنید.

</div>

In [ ]:
SYSTEM_FA = """تو نظرات فارسی مشتریان یک فروشگاه اینترنتی لوازم الکترونیکی را دسته‌بندی می‌کنی.

برچسب‌ها:
- positive: مشتری از کالا، ارسال یا خدمات راضی است.
- negative: مشتری مشکلی گزارش می‌کند یا ناراضی است.
- neutral: بیان یک واقعیت یا نظری که به‌وضوح مثبت یا منفی نیست.

فقط یکی از سه کلمهٔ positive، negative یا neutral را بنویس."""

rows = []
for language, system in [("EN", SYSTEM_EN), ("FA", SYSTEM_FA)]:
    for mode, examples in [("zero-shot", ()), ("few-shot", EXAMPLES)]:
        acc, _ = accuracy(make_classifier(system, examples))
        rows.append({"instructions": language, "mode": mode, "accuracy": round(acc, 3)})
pd.DataFrame(rows).pivot(index="mode", columns="instructions", values="accuracy")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

نتیجه به مدل بستگی دارد. با مدل‌های خانوادهٔ Qwen معمولاً دستورالعمل انگلیسی کمی پایدارتر است و Few-shot فاصلهٔ دو زبان را کم می‌کند، چون مثال‌ها قالب خروجی را صریحاً نشان می‌دهند. با مجموعهٔ ۴۸تایی، اختلاف یکی دو نمونه، یعنی حدود ۲ تا ۴ درصد، معنادار نیست. در جلسهٔ ۵ می‌بینیم چطور بازهٔ اطمینان را حساب کنیم.

به یک خطای پنهان هم دقت کنید: اگر مدل در پاسخ فارسی کلمهٔ «مثبت» بنویسد، `parse_label` آن را `unknown` می‌کند. اگر در حالت FA تعداد `unknown` زیاد است، مشکل از پرامپت نیست، از تجزیهٔ خروجی است. خروجی ساختاریافته (بخش ۶ جلسه) این مشکل را کاملاً حل می‌کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

با تحلیل جدول خطاها، تعریف برچسب `neutral` را دقیق‌تر کنید و دو مثال Few-shot جدید بنویسید. دقت چقدر تغییر کرد؟

</div>

In [ ]:
SYSTEM_V2 = SYSTEM_EN.replace(
    "- neutral: a factual statement or an opinion that is neither clearly positive nor negative.",
    "- neutral: a factual statement (contents of the box, weight, compatibility, price change), "
    "a report that something happened as expected, a mixed opinion where good and bad points are "
    "balanced, or a review written before the customer has used the product.",
)
EXAMPLES_V2 = EXAMPLES + [
    ("سفارش طبق زمان اعلام‌شده تحویل شد.", "neutral"),
    ("دوربینش خوبه ولی باتریش زود تموم میشه، در کل معمولیه.", "neutral"),
]

acc_v1, pred_v1 = accuracy(make_classifier(SYSTEM_EN, EXAMPLES))
acc_v2, pred_v2 = accuracy(make_classifier(SYSTEM_V2, EXAMPLES_V2))
print(f"few-shot v1: {acc_v1:.2%}   few-shot v2: {acc_v2:.2%}")

changed = reviews.assign(v1=pred_v1, v2=pred_v2).query("v1 != v2")
changed[["text", "label", "v1", "v2"]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

جدول آخر نمونه‌هایی را نشان می‌دهد که پیش‌بینی‌شان عوض شده است. هم نمونه‌هایی را ببینید که درست شده‌اند و هم آن‌هایی که **خراب شده‌اند**. تعریف گسترده‌تر `neutral` ممکن است نظرات مثبت ملایم را هم خنثی کند.

**یک هشدار روش‌شناختی:** ما تعریف را با نگاه کردن به خطاهای همین مجموعه بهتر کردیم و دوباره روی همان مجموعه دقت گرفتیم. این عدد **خوش‌بینانه** است، چون پرامپت را روی دادهٔ آزمون تنظیم کرده‌ایم. در کار واقعی، داده را به Dev و Test تقسیم کنید، روی Dev تحلیل خطا کنید و فقط در پایان روی Test گزارش بدهید (جلسهٔ ۵).

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

فیلد `battery_health` (درصد سلامت باتری، یا `null`) را به `PhoneAd` اضافه کنید و پنج آگهی جدید با شکل‌های مختلف نوشتن قیمت بسازید.

</div>

In [ ]:
class PhoneAd(BaseModel):
    brand: str = Field(description="Brand in English, e.g. Apple, Samsung, Xiaomi")
    model: str = Field(description="Model name in English")
    storage_gb: Optional[int] = Field(description="Storage in GB, or null")
    price_toman: Optional[int] = Field(description="Price in toman as an integer, or null if negotiable")
    city: Optional[str] = Field(description="City name in Persian, or null")
    condition: Literal["new", "like_new", "used"]
    battery_health: Optional[int] = Field(description="Battery health in percent (0-100), or null if not mentioned")


SYSTEM_EXTRACT = (
    "You extract structured data from Persian second-hand phone ads. "
    "'میلیون' means x1,000,000 and 'هزار' or 'تومن' after a small number means x1,000. "
    "'تومان' after a large number is the exact amount. "
    "condition: new = sealed/آکبند/پلمب, like_new = در حد نو, used = کارکرده or سالم. "
    "battery_health is the battery percentage if mentioned (e.g. 'باتری ۹۲'), otherwise null."
)

ads = [
    "آیفون ۱۴ پرو ۲۵۶ گیگ، باتری ۹۲ درصد، در حد نو، ۴۸ میلیون و پونصد، تهران",
    "سامسونگ S23 کارکرده سالم، سلامت باتری ۸۵٪، قیمت: 27,000,000 تومان، رشت",
    "شیائومی ۱۳ تی ۲۵۶ گیگ آکبند، بیست و دو میلیون، قم",
    "گوشی ساده نوکیا، باتری نو، ۱ میلیون و ۲۰۰، اراک",
    "پیکسل ۸ گوگل ۱۲۸ گیگ، باتری 79، قیمت مقطوع ۳۱.۵ میلیون، کرج",
]
pd.DataFrame([extract(PhoneAd, SYSTEM_EXTRACT, ad).model_dump() for ad in ads])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

موارد دشوار عمداً متنوع‌اند:

| آگهی | نکتهٔ دشوار | مقدار درست |
|---|---|---|
| ۱ | «۴۸ میلیون و پونصد» به زبان محاوره | ۴۸٬۵۰۰٬۰۰۰ |
| ۲ | قیمت با ارقام انگلیسی و جداکننده | ۲۷٬۰۰۰٬۰۰۰ |
| ۳ | قیمت با حروف | ۲۲٬۰۰۰٬۰۰۰ |
| ۴ | «باتری نو» درصد ندارد؛ «۱ میلیون و ۲۰۰» یعنی ۱٬۲۰۰٬۰۰۰ | battery_health: null |
| ۵ | «باتری 79» بدون علامت درصد | ۷۹ و ۳۱٬۵۰۰٬۰۰۰ |

مدل‌های کوچک بیشتر در «میلیون و پونصد» و قیمت حروفی اشتباه می‌کنند. اگر محاسبهٔ عدد مهم است، می‌توان از مدل خواست قیمت را **همان‌طور که نوشته شده** به صورت رشته استخراج کند و تبدیل به عدد را با کد، مثلاً تابع `parse_price` جلسهٔ ۲، انجام داد. تقسیم کار بین مدل و کد معمولاً دقیق‌تر از سپردن همه چیز به مدل است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

یک Schema به نام `OrderRequest` با فیلدهای `product`، `quantity`، `color` و `city` تعریف کنید و پیام‌هایی مثل «دو تا گوشی سامسونگ A55 آبی برای شیراز می‌خوام» را استخراج کنید. همین Schema را در جلسه ۱۰ برای فاین‌تیون استفاده می‌کنیم.

</div>

In [ ]:
class OrderRequest(BaseModel):
    product: str = Field(description="The product name as written by the customer")
    quantity: int = Field(description="Number of items; 1 if not mentioned")
    color: Optional[str] = Field(description="Colour in Persian, or null if not mentioned")
    city: Optional[str] = Field(description="Destination city in Persian, or null if not mentioned")


SYSTEM_ORDER = ("You extract an order from a Persian customer message. Persian number words such as "
                "'یه', 'دو تا' and 'سه عدد' are quantities. Return null for missing fields.")

with open(download_data("orders_real.jsonl"), encoding="utf-8") as f:
    orders = [json.loads(line) for line in f]

rows = []
for item in orders:
    pred = extract(OrderRequest, SYSTEM_ORDER, item["message"])
    gold = item["order"]
    rows.append({"message": item["message"], "product": pred.product,
                 "quantity_ok": pred.quantity == gold["quantity"],
                 "color_ok": pred.color == gold["color"], "city_ok": pred.city == gold["city"]})
result = pd.DataFrame(rows)
print(result[["quantity_ok", "color_ok", "city_ok"]].mean().round(2).to_dict())
result

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

فایل `orders_real.jsonl` دوازده پیام سفارش با پاسخ درست دارد که در جلسهٔ ۱۰ هم استفاده می‌شود. اینجا فیلد `product` را فقط نمایش دادیم و ارزیابی نکردیم، چون مدل نام کالا را همان‌طور که مشتری نوشته برمی‌گرداند، ولی برای ثبت سفارش به **شناسهٔ کالا** در فهرست فروشگاه (`P1001` و ...) نیاز داریم.

برای نگاشت نام به شناسه دو راه داریم: فهرست کالاها را در پرامپت بگذاریم، که پرامپت را طولانی و گران می‌کند، یا مدل را روی این کار **فاین‌تیون** کنیم تا فهرست را «یاد بگیرد». مقایسهٔ این دو راه موضوع جلسهٔ ۱۰ است.

یک نکتهٔ کوچک ولی مهم: مقایسهٔ `pred.color == gold["color"]` برای «نقره‌ای» به نیم‌فاصله حساس است. اگر مدل «نقره ای» با فاصله بنویسد، خطا شمرده می‌شود. پیش از مقایسه، هر دو را نرمال کنید (جلسهٔ ۷).

</div>